In [5]:
from datasets import load_dataset

dataset = load_dataset(
    "nlpaueb/finer-139",
    revision="refs/convert/parquet"
)

print(dataset)

C:\Users\bader\Documents\projects\financial-document-analysis\.venv\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\bader\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating train split: 900384 examples [00:01, 588047.26 examples/s]
Generating validation split: 112494 exam

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 900384
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 112494
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 108378
    })
})


In [33]:
# 1. Look at the very first example in the training set
sample = dataset["train"][0]

print("Tokens:", sample["tokens"])
print("NER Tags (Integers):", sample["ner_tags"])

# Let's verify that every token has exactly one tag
print(f"\nLength of tokens: {len(sample['tokens'])}")
print(f"Length of tags: {len(sample['ner_tags'])}")

# 2. Extract the label mapping (the human-readable names of the tags)
label_names = dataset["train"].features["ner_tags"].feature.names
print(f"\nTotal number of unique labels: {len(label_names)}")

# Let's print the first 15 labels to see how they are formatted
print("First 15 labels:")
for i, label in enumerate(label_names[:15]):
    print(f"{i}: {label}")

Tokens: ['ITEM', '1', 'Financial', 'Statements', 'Lennar', 'Corporation', 'and', 'Subsidiaries', 'Condensed', 'Consolidated', 'Balance', 'Sheets', '(', 'Dollars', 'in', 'thousands', ',', 'except', 'shares', 'and', 'per', 'share', 'amounts', ')', '(', 'unaudited', ')', '(', '1', ')', 'Under', 'certain', 'provisions', 'of', 'Accounting', 'Standards', 'Codification', '(', '“', 'ASC', '”', ')', 'Topic', '810', ',', 'Consolidations', ',', '(', '“', 'ASC', '810', '”', ')', 'the', 'Company', 'is', 'required', 'to', 'separately', 'disclose', 'on', 'its', 'condensed', 'consolidated', 'balance', 'sheets', 'the', 'assets', 'owned', 'by', 'consolidated', 'variable', 'interest', 'entities', '(', '“', 'VIEs', '”', ')', 'and', 'liabilities', 'of', 'consolidated', 'VIEs', 'as', 'to', 'which', 'neither', 'Lennar', 'Corporation', ',', 'or', 'any', 'of', 'its', 'subsidiaries', ',', 'has', 'any', 'obligations', '.', 'As', 'of', 'May', '31', ',', '2016', ',', 'total', 'assets', 'include', '$', '645.1', 'mi

In [38]:
# Print tokens and their labels, ignoring the 'O' tags to keep the output clean
print("Extracted Entities in Example 1:\n")

for token, tag_int in zip(sample['tokens'], sample['ner_tags']):
    tag_name = label_names[tag_int]
    
    if tag_name != "O":
        print(f"{token} -> {tag_name}")

Extracted Entities in Example 1:

4.5 -> B-EquityMethodInvestments
35.8 -> B-EquityMethodInvestments


In [41]:
from collections import Counter
import itertools

print("Counting tags across 900,000+ sentences...")

# itertools.chain.from_iterable efficiently flattens the list of lists into one giant sequence
all_tags = itertools.chain.from_iterable(dataset['train']['ner_tags'])

# Count the occurrences of each integer tag
tag_counts = Counter(all_tags)

# Get the total number of non-O tags to see how many entities we have in total
total_entities = sum(count for tag_int, count in tag_counts.items() if label_names[tag_int] != "O")
print(f"\nTotal entities annotated in training set: {total_entities:,}")

print("\n--- Top 25 Most Frequent Entities ---")
# We pull top 25 just in case some are 'I-' tags, to make sure we see a good variety
for tag_int, count in tag_counts.most_common(25):
    tag_name = label_names[tag_int]
    if tag_name != "O" and tag_name.startswith("B-"):
        print(f"{tag_name}: {count:,}")

Counting tags across 900,000+ sentences...

Total entities annotated in training set: 318,798

--- Top 25 Most Frequent Entities ---
B-DebtInstrumentInterestRateStatedPercentage: 15,208
B-LineOfCreditFacilityMaximumBorrowingCapacity: 11,693
B-DebtInstrumentBasisSpreadOnVariableRate1: 11,445
B-DebtInstrumentFaceAmount: 10,749
B-AllocatedShareBasedCompensationExpense: 8,710
B-AntidilutiveSecuritiesExcludedFromComputationOfEarningsPerShareAmount: 7,711
B-EffectiveIncomeTaxRateContinuingOperations: 7,379
B-AmortizationOfIntangibleAssets: 6,421
B-ConcentrationRiskPercentage1: 5,456
B-ShareBasedCompensationArrangementByShareBasedPaymentAwardEquityInstrumentsOtherThanOptionsGrantsInPeriod: 4,817
B-EquityMethodInvestmentOwnershipPercentage: 4,545
B-IncomeTaxExpenseBenefit: 4,530
B-RestructuringCharges: 4,503
B-OperatingLeaseRightOfUseAsset: 3,908
B-LineOfCredit: 3,601
B-EmployeeServiceShareBasedCompensationNonvestedAwardsTotalCompensationCostNotYetRecognizedPeriodForRecognition1: 3,554
B-Reven

In [42]:
import numpy as np

# 1. Compute token length for each sentence in the training set
token_lengths = [len(tokens) for tokens in dataset['train']['tokens']]

# 2. Count how many entities are in each sentence (sum of non-zero tags)
entities_per_sentence = [
    sum(1 for tag in tags if tag != 0) 
    for tags in dataset['train']['ner_tags']
]

# Compute statistics
print("=== Sentence Length (Tokens) Statistics ===")
print(f"Min length:    {np.min(token_lengths)}")
print(f"Mean length:   {np.mean(token_lengths):.2f}")
print(f"Median length: {np.median(token_lengths)}")
print(f"95th percentile: {np.percentile(token_lengths, 95):.1f}")
print(f"99th percentile: {np.percentile(token_lengths, 99):.1f}")
print(f"Max length:    {np.max(token_lengths)}")

# Compute sparsity statistics
sentences_with_entities = sum(1 for count in entities_per_sentence if count > 0)
total_sentences = len(entities_per_sentence)
pct_with_entities = (sentences_with_entities / total_sentences) * 100

print("\n=== Entity Distribution Across Sentences ===")
print(f"Total sentences: {total_sentences:,}")
print(f"Sentences with at least 1 entity: {sentences_with_entities:,} ({pct_with_entities:.2f}%)")
print(f"Sentences with 0 entities: {total_sentences - sentences_with_entities:,} ({100 - pct_with_entities:.2f}%)")

=== Sentence Length (Tokens) Statistics ===
Min length:    2
Mean length:   45.34
Median length: 37.0
95th percentile: 104.0
99th percentile: 171.0
Max length:    1260

=== Entity Distribution Across Sentences ===
Total sentences: 900,384
Sentences with at least 1 entity: 179,195 (19.90%)
Sentences with 0 entities: 721,189 (80.10%)


In [60]:
b_tag_counts = {
    label_names[tag_int]: count 
    for tag_int, count in tag_counts.items() 
    if label_names[tag_int].startswith("B-")
}


rarest_5 = sorted(b_counts.items(), key=lambda item: item[1])[:5]
print(rarest_5)

[('B-ProceedsFromIssuanceOfCommonStock', 794), ('B-SaleOfStockPricePerShare', 800), ('B-CommonStockSharesOutstanding', 806), ('B-PreferredStockDividendRatePercentage', 832), ('B-OperatingLeasePayments', 835)]


In [71]:
def inspect_sentence(index, split="train"):
    sample = dataset[split][index]
    tokens = sample["tokens"]
    tags = sample["ner_tags"]
    
    annotated_tokens = []
    
    for token, tag_id in zip(tokens, tags):
        label = label_names[tag_id]
        if label == "O":
            annotated_tokens.append(token)
        else:
            # Strip BIO prefix for cleaner reading
            clean_label = label.replace("B-", "").replace("I-", "")
            annotated_tokens.append(f"[{token} -> {clean_label}]")
            
    print(f"=== Sentence #{index} ({split}) ===\n\n")
    print(" ".join(annotated_tokens))

In [72]:
inspect_sentence(0)

=== Sentence #0 (train) ===


ITEM 1 Financial Statements Lennar Corporation and Subsidiaries Condensed Consolidated Balance Sheets ( Dollars in thousands , except shares and per share amounts ) ( unaudited ) ( 1 ) Under certain provisions of Accounting Standards Codification ( “ ASC ” ) Topic 810 , Consolidations , ( “ ASC 810 ” ) the Company is required to separately disclose on its condensed consolidated balance sheets the assets owned by consolidated variable interest entities ( “ VIEs ” ) and liabilities of consolidated VIEs as to which neither Lennar Corporation , or any of its subsidiaries , has any obligations . As of May 31 , 2016 , total assets include $ 645.1 million related to consolidated VIEs of which $ 8.2 million is included in Lennar Homebuilding cash and cash equivalents , $ 0.1 million in Lennar Homebuilding receivables , net , $ 6.2 million in Lennar Homebuilding finished homes and construction in progress , $ 158.8 million in Lennar Homebuilding land and land under